# Horn model evaluation plots

3つの評価フォルダ

- `Horn_fullmodel_evaluated`
- `Horn_cutmodel_evaluated`
- `Horn_lightmodel_evaluated`

から `S11`, `Beam`, `Ellipticity`, `XPD` のみを読み込み、Matplotlibで比較プロットを作成します。

- ファイルI/O: `pathlib.Path`
- Plot: `fig, ax = plt.subplots(...)`
- 色:
  - Full: black
  - Quater cut: red
  - Quater Poly: blue

> `ROOT` は3つのフォルダが置かれている親ディレクトリに変更してください。


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# Path settings
# ============================================================
# 例:
# ROOT = Path(r"C:\Users\yourname\Desktop\horn_results")
ROOT = Path.cwd()

MODELS = {
    "full": {
        "folder": ROOT / "Horn_fullmodel_evaluated",
        "color": "black",
        "label": "Full",
    },
    "cut": {
        "folder": ROOT / "Horn_cutmodel_evaluated",
        "color": "red",
        "label": "Quater cut",
    },
    "light": {
        "folder": ROOT / "Horn_lightmodel_evaluated",
        "color": "blue",
        "label": "Quater Poly",
    },
}

TARGET_FILES = ("S11", "Beam", "Ellipticity", "XPD")


## ファイル探索・読み込み

Windowsで拡張子表示がOFFの場合などを考慮して、以下のどちらでも読めるようにしています。

- `S11`, `Beam`, `Ellipticity`, `XPD` のように拡張子なし
- `S11.csv`, `Beam.txt` のように拡張子あり

ファイル名の **stem** が対象名と一致するものを使用します。


In [ ]:
def find_data_file(folder: Path, stem: str) -> Path:
    """folder内から、ファイル名またはstemが指定名に一致するファイルを探す。"""
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    # まず完全一致（拡張子なし）を確認
    exact = folder / stem
    if exact.is_file():
        return exact

    # 拡張子ありの場合にも対応
    matches = [
        p for p in folder.iterdir()
        if p.is_file() and p.stem.lower() == stem.lower()
    ]

    if not matches:
        raise FileNotFoundError(
            f"'{stem}' was not found in: {folder}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple files matched '{stem}' in {folder}: {matches}"
        )

    return matches[0]


def read_data_file(path: Path) -> pd.DataFrame:
    """CSV形式の評価データをDataFrameとして読み込む。"""
    return pd.read_csv(path, encoding="utf-8-sig")


def load_all_data(models: dict, target_files: tuple[str, ...]) -> dict:
    """全モデルについて対象ファイルだけを読み込む。"""
    data = {}

    for model_key, info in models.items():
        folder = info["folder"]
        data[model_key] = {}

        for stem in target_files:
            file_path = find_data_file(folder, stem)
            data[model_key][stem] = read_data_file(file_path)

            print(
                f"{model_key:>5} | {stem:<12} | "
                f"{file_path.name:<20} | shape={data[model_key][stem].shape}"
            )

    return data


data = load_all_data(MODELS, TARGET_FILES)


## 読み込んだデータの確認

必要なら下のセルで列名や先頭行を確認できます。


In [ ]:
for model_key in MODELS:
    print(f"\n===== {MODELS[model_key]['label']} =====")
    for name in TARGET_FILES:
        print(f"\n--- {name} ---")
        display(data[model_key][name].head())


## S11

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["S11"]

    x = df.iloc[:, 0]
    y = df.iloc[:, 1]

    ax.plot(
        x,
        y,
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("S11 [dB]")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## Beam

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["Beam"]

    theta = df.iloc[:, 0]
    gain = df.iloc[:, 1]

    ax.plot(
        theta,
        gain,
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.set_xlabel("Theta [deg]")
ax.set_ylabel("GainTotal / PeakGain [-]")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## Ellipticity

各モデルの色は指定通り固定し、`Phi=0 deg` と `Phi=90 deg` は線種で区別します。

- solid: Phi = 0 deg
- dashed: Phi = 90 deg


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["Ellipticity"]

    freq = df.iloc[:, 0]
    phi0 = df.iloc[:, 1]
    phi90 = df.iloc[:, 2]

    ax.plot(
        freq,
        phi0,
        color=info["color"],
        linestyle="-",
        linewidth=1.8,
    )
    ax.plot(
        freq,
        phi90,
        color=info["color"],
        linestyle="--",
        linewidth=1.8,
    )

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("Beam width at 0.5 [deg]")
ax.grid(True, alpha=0.3)

# モデル色の凡例
model_handles = [
    Line2D(
        [0], [0],
        color=info["color"],
        linewidth=2,
        label=info["label"],
    )
    for info in MODELS.values()
]
legend_models = ax.legend(
    handles=model_handles,
    title="Model",
    loc="best",
)
ax.add_artist(legend_models)

# Phiの線種の凡例
phi_handles = [
    Line2D([0], [0], color="gray", linestyle="-", linewidth=2, label="Phi = 0 deg"),
    Line2D([0], [0], color="gray", linestyle="--", linewidth=2, label="Phi = 90 deg"),
]
ax.legend(
    handles=phi_handles,
    title="Cut plane",
    loc="upper right",
)

fig.tight_layout()
plt.show()


## XPD

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["XPD"]

    freq = df.iloc[:, 0]
    xpd = df.iloc[:, 1]

    ax.plot(
        freq,
        xpd,
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("XPD [dB]")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## Optional: 図を保存する場合

必要なら各plotセルで `plt.show()` の前に、例えば以下を追加してください。

```python
OUTPUT_DIR = ROOT / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "S11.png", dpi=300, bbox_inches="tight")
```
